<a href="https://colab.research.google.com/github/helenpilreena2003-tech/-Real-Time-Handwritten-Digit-Recognition-System/blob/main/handwriting_finder.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import tensorflow as tf
import matplotlib.pyplot as plt
import numpy as np
import random

print("⏳ Step 1: Downloading the MNIST dataset...")
mnist = tf.keras.datasets.mnist
(x_train, y_train), (x_test, y_test) = mnist.load_data()
print("✅ Dataset downloaded successfully!")

⏳ Step 1: Downloading the MNIST dataset...
11490434/11490434 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
✅ Dataset downloaded successfully!


In [ ]:
print("🧹 Step 2: Cleaning and normalizing the image data...")
# Convert pixel values from 0-255 to a scale of 0.0 to 1.0 for the AI
x_train = x_train / 255.0
x_test = x_test / 255.0
print("✅ Data normalization complete!")

🧹 Step 2: Cleaning and normalizing the image data...
✅ Data normalization complete!


In [ ]:
print("🧠 Step 3: Building the Deep Learning Neural Network...")
model = tf.keras.models.Sequential([
  tf.keras.layers.Flatten(input_shape=(28, 28)),          # Flatten the 28x28 image into a line
  tf.keras.layers.Dense(128, activation='relu'),          # Hidden layer with 128 neurons
  tf.keras.layers.Dropout(0.2),                           # Prevent overfitting
  tf.keras.layers.Dense(10, activation='softmax')         # Output layer for digits 0-9
])

print("⚙️ Step 4: Compiling the model...")
model.compile(optimizer='adam',
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])
print("✅ Model is built and ready to learn!")

🧠 Step 3: Building the Deep Learning Neural Network...
⚙️ Step 4: Compiling the model...
✅ Model is built and ready to learn!


/usr/local/lib/python3.12/dist-packages/keras/src/layers/reshaping/flatten.py:37: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


In [ ]:
print("🚀 Step 5: Training the Deep Learning model...")
model.fit(x_train, y_train, epochs=5)
print("\n✅ Model Training SUCCESS!")

🚀 Step 5: Training the Deep Learning model...
Epoch 1/5
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 8s 3ms/step - accuracy: 0.9132 - loss: 0.2970
Epoch 2/5
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 7s 4ms/step - accuracy: 0.9555 - loss: 0.1462
Epoch 3/5
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 6s 3ms/step - accuracy: 0.9668 - loss: 0.1085
Epoch 4/5
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 7s 4ms/step - accuracy: 0.9722 - loss: 0.0894
Epoch 5/5
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 10s 4ms/step - accuracy: 0.9762 - loss: 0.0751

✅ Model Training SUCCESS!


In [1]:
!pip install gradio==3.50.0 opencv-python-headless

import gradio as gr
import numpy as np
import cv2 # OpenCV for advanced image processing

print("🚀 Launching the Ultimate Accuracy Web App with OpenCV Auto-Centering...")

custom_css = """
canvas { cursor: url('https://cdn-icons-png.flaticon.com/24/1250/1250615.png') 0 24, crosshair !important; }
.gradio-container { background-color: #f0f2f5 !important; }
#center-box {
    max-width: 400px !important; margin: 30px auto !important;
    border: 3px dashed #ff9900 !important; border-radius: 12px;
    padding: 20px; background-color: #ffffff; box-shadow: 0 4px 6px rgba(0, 0, 0, 0.1);
}
"""

def predict_digit(img):
    if img is None:
        return {"Please draw a number": 1.0}

    # 1. Invert the image (Black drawing on White bg -> White drawing on Black bg)
    img = 255 - img

    # 2. Find where the drawing actually is (The Bounding Box)
    coords = cv2.findNonZero(img)
    if coords is None:
        return {"Please draw a number": 1.0}

    x, y, w, h = cv2.boundingRect(coords)

    # 3. Crop out the empty space so we only have the drawn digit
    cropped = img[y:y+h, x:x+w]

    # 4. Resize the cropped digit to fit perfectly inside a 20x20 box (preserving shape)
    if w > h:
        new_w = 20
        new_h = int((20 / w) * h)
    else:
        new_h = 20
        new_w = int((20 / h) * w)

    resized = cv2.resize(cropped, (new_w, new_h), interpolation=cv2.INTER_AREA)

    # 5. Place the resized digit exactly into the center of a 28x28 black canvas
    padded_img = np.zeros((28, 28), dtype=np.uint8)
    y_offset = (28 - new_h) // 2
    x_offset = (28 - new_w) // 2
    padded_img[y_offset:y_offset+new_h, x_offset:x_offset+new_w] = resized

    # 6. Normalize and predict
    final_img = padded_img / 255.0
    final_img = final_img.reshape(1, 28, 28)

    prediction = model.predict(final_img, verbose=0)[0]
    return {str(i): float(prediction[i]) for i in range(10)}

# ==========================================
# ADVANCED UI DESIGN (GRADIO BLOCKS)
# ==========================================
with gr.Blocks(css=custom_css, theme=gr.themes.Soft()) as interface:
    gr.Markdown("<h1 style='text-align: center; color: #333;'>✏️ Ultimate AI Predictor (OpenCV Enabled)</h1>")
    gr.Markdown("<h3 style='text-align: center; color: #666;'>Draw anywhere in the box. The AI will auto-crop, center, and predict with max accuracy!</h3>")

    with gr.Row():
        with gr.Column(elem_id="center-box"):
            # We take a larger high-res input (280x280) so the user's drawing is smooth,
            # and our OpenCV code will automatically shrink it to 28x28 perfectly!
            img_input = gr.Image(shape=(280, 280), image_mode="L", invert_colors=False, source="canvas", brush_radius=15, label="Draw Here ✏️")
            predict_btn = gr.Button("🤖 Predict Number", variant="primary")
            output_label = gr.Label(num_top_classes=3, label="AI Prediction Result")

    predict_btn.click(fn=predict_digit, inputs=img_input, outputs=output_label)

interface.launch(share=True)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 3.2 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of opencv-python-headless to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.3/20.3 MB 76.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 299.2/299.2 kB 26.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.0/50.0 MB 18.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 85.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 119.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 118.1/118.1 kB 12.2 MB/s eta 0:00:00
  Attempting uninstall: websockets
    Found existing installation: websockets 15.0.1
    Uninstalling websockets-15.0.1:
      Successfully uninstalled websockets-15.0.1
  Attempting uninstall: pillow
    Found existing installation: pillow 11.3.0
    Uninstalling pillow-11.3.0:
   

ValueError: numpy.dtype size changed, may indicate binary incompatibility. Expected 96 from C header, got 88 from PyObject